In [ ]:
import os
import io
import pickle
import boto3
import pandas as pd
import numpy as np

#### Constants

In [ ]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

LIST_MAIN_TYPES = ['FF', 'SL', 'SI', 'FT', 'CH', 'CU', 'FC', 'FS']

s3 = boto3.client('s3')

#### Functions

In [ ]:
def save_pickle_to_s3(obj, bucket, key):
    """Serialize a Python object and upload to S3."""
    buffer = io.BytesIO()
    pickle.dump(obj, buffer)
    buffer.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    print(f'Saved to s3://{bucket}/{key}')

#### Import Data from S3

In [ ]:
df_train = pd.read_csv(f's3://{str_bucket}/02_data_split/train.csv')
df_valid = pd.read_csv(f's3://{str_bucket}/02_data_split/valid.csv')
df_test = pd.read_csv(f's3://{str_bucket}/02_data_split/test.csv')

print(f'Train: {df_train.shape}')
print(f'Valid: {df_valid.shape}')
print(f'Test:  {df_test.shape}')

#### Feature Engineering Functions

In [ ]:
def create_situation_features(df):
    """Create game situation features from available-prior-to-pitch columns."""
    df = df.copy()
    
    # score differential from pitching team perspective
    # if top=1, pitching team is home: home_runs - away_runs
    # if top=0, pitching team is away: away_runs - home_runs
    df['score_diff'] = np.where(
        df['top'] == 1,
        df['home_team_runs'] - df['away_team_runs'],
        df['away_team_runs'] - df['home_team_runs']
    )
    
    # runners on base
    df['on_1b_flag'] = df['on_1b'].notna().astype(int)
    df['on_2b_flag'] = df['on_2b'].notna().astype(int)
    df['on_3b_flag'] = df['on_3b'].notna().astype(int)
    df['runners_on'] = df['on_1b_flag'] + df['on_2b_flag'] + df['on_3b_flag']
    
    # handedness matchup
    df['same_hand'] = (df['p_throws'] == df['stand']).astype(int)
    
    # is first pitch of at-bat
    df['is_first_pitch'] = (df['pcount_at_bat'] == 1).astype(int)
    
    return df


def create_lag_features(df):
    """Create previous-pitch-in-at-bat lag features."""
    df = df.copy()
    df = df.sort_values(['game_pk', 'at_bat_num', 'pcount_at_bat'])
    
    # previous pitch type in this at-bat
    df['prev_pitch_type'] = df.groupby(['game_pk', 'at_bat_num'])['pitch_type'].shift(1)
    df['prev_pitch_type'] = df['prev_pitch_type'].fillna('NONE')
    
    # 2nd previous pitch type in this at-bat
    df['prev_pitch_type_2'] = df.groupby(['game_pk', 'at_bat_num'])['pitch_type'].shift(2)
    df['prev_pitch_type_2'] = df['prev_pitch_type_2'].fillna('NONE')
    
    return df


def compute_pitcher_stats(df_source):
    """Compute pitcher-level pitch mix statistics from a source dataframe.
    Returns dict of DataFrames to merge onto any split."""
    stats = {}
    
    # 1. overall pitch type distribution per pitcher
    df_pitcher_mix = pd.crosstab(df_source['pitcher_id'], df_source['pitch_type'], normalize='index')
    df_pitcher_mix.columns = [f'pitcher_pct_{c}' for c in df_pitcher_mix.columns]
    stats['pitcher_mix'] = df_pitcher_mix
    
    # 2. pitch type distribution per pitcher per count
    df_source = df_source.copy()
    df_source['count_str'] = df_source['balls'].astype(str) + '-' + df_source['strikes'].astype(str)
    df_pitcher_count = pd.crosstab(
        [df_source['pitcher_id'], df_source['count_str']],
        df_source['pitch_type'],
        normalize='index'
    )
    df_pitcher_count.columns = [f'pitcher_count_pct_{c}' for c in df_pitcher_count.columns]
    stats['pitcher_count_mix'] = df_pitcher_count
    
    # 3. total pitches thrown by pitcher (volume)
    stats['pitcher_volume'] = df_source.groupby('pitcher_id').size().rename('pitcher_total_pitches')
    
    # 4. pitcher repertoire size
    stats['pitcher_n_types'] = df_source.groupby('pitcher_id')['pitch_type'].nunique().rename('pitcher_n_types')
    
    return stats


def apply_pitcher_stats(df, stats):
    """Merge precomputed pitcher stats onto a dataframe."""
    df = df.copy()
    df['count_str'] = df['balls'].astype(str) + '-' + df['strikes'].astype(str)
    
    # overall pitcher mix
    df = df.merge(stats['pitcher_mix'], left_on='pitcher_id', right_index=True, how='left')
    
    # pitcher mix per count
    df = df.merge(stats['pitcher_count_mix'], left_on=['pitcher_id', 'count_str'], right_index=True, how='left')
    
    # pitcher volume and repertoire
    df = df.merge(stats['pitcher_volume'], left_on='pitcher_id', right_index=True, how='left')
    df = df.merge(stats['pitcher_n_types'], left_on='pitcher_id', right_index=True, how='left')
    
    return df

#### Compute Pitcher Stats from Train Set Only

To prevent data leakage, all pitcher-level statistics are computed from the training set and then applied to all splits.

In [ ]:
dict_pitcher_stats = compute_pitcher_stats(df_train)

print(f'Pitcher mix shape: {dict_pitcher_stats["pitcher_mix"].shape}')
print(f'Pitcher-count mix shape: {dict_pitcher_stats["pitcher_count_mix"].shape}')
print(f'Unique pitchers in train: {len(dict_pitcher_stats["pitcher_volume"])}')

#### Apply Feature Engineering to All Splits

In [ ]:
list_dfs = []
for str_name, df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    print(f'Processing {str_name}...')
    df_proc = create_situation_features(df_split)
    df_proc = create_lag_features(df_proc)
    df_proc = apply_pitcher_stats(df_proc, dict_pitcher_stats)
    list_dfs.append(df_proc)
    print(f'  {str_name} shape after features: {df_proc.shape}')

df_train_proc, df_valid_proc, df_test_proc = list_dfs

#### Check for Pitchers in Valid/Test Not Seen in Train

In [ ]:
set_train_pitchers = set(df_train['pitcher_id'].unique())
set_valid_pitchers = set(df_valid['pitcher_id'].unique())
set_test_pitchers = set(df_test['pitcher_id'].unique())

int_valid_unseen = len(set_valid_pitchers - set_train_pitchers)
int_test_unseen = len(set_test_pitchers - set_train_pitchers)

print(f'Pitchers in valid not in train: {int_valid_unseen}')
print(f'Pitchers in test not in train: {int_test_unseen}')
print(f'\nThese will have NaN for pitcher stats features — filled with 0 below.')

#### Select Final Features and Encode

In [ ]:
# numeric features
list_numeric = [
    'inning', 'top', 'outs', 'balls', 'strikes', 'fouls',
    'pcount_at_bat', 'pcount_pitcher',
    'score_diff',
    'on_1b_flag', 'on_2b_flag', 'on_3b_flag', 'runners_on',
    'same_hand', 'is_first_pitch',
    'pitcher_total_pitches', 'pitcher_n_types',
]

# pitcher mix features (dynamic column names)
list_pitcher_mix = [c for c in df_train_proc.columns if c.startswith('pitcher_pct_')]
list_pitcher_count_mix = [c for c in df_train_proc.columns if c.startswith('pitcher_count_pct_')]

# categorical features to one-hot encode
list_categorical = ['p_throws', 'stand', 'prev_pitch_type', 'prev_pitch_type_2']

print(f'Numeric features: {len(list_numeric)}')
print(f'Pitcher mix features: {len(list_pitcher_mix)}')
print(f'Pitcher-count mix features: {len(list_pitcher_count_mix)}')
print(f'Categorical features: {len(list_categorical)}')

In [ ]:
def prepare_features(df, list_numeric, list_pitcher_mix, list_pitcher_count_mix, list_categorical):
    """Select and encode features, return X and y."""
    df = df.copy()
    
    # target
    y = df['pitch_type'].copy()
    
    # numeric
    X_num = df[list_numeric].copy()
    
    # pitcher mix (fill NaN with 0 for unseen pitchers)
    list_mix_cols = list_pitcher_mix + list_pitcher_count_mix
    list_mix_present = [c for c in list_mix_cols if c in df.columns]
    X_mix = df[list_mix_present].fillna(0).copy()
    
    # categorical -> one-hot
    X_cat = pd.get_dummies(df[list_categorical], columns=list_categorical, dtype=int)
    
    # combine
    X = pd.concat([X_num, X_mix, X_cat], axis=1)
    
    # fill remaining NaN with 0
    X = X.fillna(0)
    
    return X, y

In [ ]:
X_train, y_train = prepare_features(df_train_proc, list_numeric, list_pitcher_mix, list_pitcher_count_mix, list_categorical)
X_valid, y_valid = prepare_features(df_valid_proc, list_numeric, list_pitcher_mix, list_pitcher_count_mix, list_categorical)
X_test, y_test = prepare_features(df_test_proc, list_numeric, list_pitcher_mix, list_pitcher_count_mix, list_categorical)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

In [ ]:
# align columns across splits (valid/test may have different one-hot columns)
list_all_cols = sorted(set(X_train.columns) | set(X_valid.columns) | set(X_test.columns))

X_train = X_train.reindex(columns=list_all_cols, fill_value=0)
X_valid = X_valid.reindex(columns=list_all_cols, fill_value=0)
X_test = X_test.reindex(columns=list_all_cols, fill_value=0)

print(f'Aligned feature count: {len(list_all_cols)}')
print(f'X_train: {X_train.shape}')
print(f'X_valid: {X_valid.shape}')
print(f'X_test:  {X_test.shape}')

In [ ]:
# verify no data leakage: check columns used
print('Feature columns:')
for col in sorted(X_train.columns):
    print(f'  {col}')

#### Save Processed Data to S3

In [ ]:
# save feature matrices and targets to S3
for str_name, df_x, sr_y in [('train', X_train, y_train), ('valid', X_valid, y_valid), ('test', X_test, y_test)]:
    df_x.to_csv(f's3://{str_bucket}/{str_task}/X_{str_name}.csv', index=False)
    sr_y.to_csv(f's3://{str_bucket}/{str_task}/y_{str_name}.csv', index=False)
    print(f'Saved X_{str_name}.csv ({df_x.shape}) and y_{str_name}.csv ({sr_y.shape}) to S3')

# save pitcher stats pickle to S3
save_pickle_to_s3(dict_pitcher_stats, str_bucket, f'{str_task}/pitcher_stats.pkl')

# save feature column list to S3
save_pickle_to_s3(list_all_cols, str_bucket, f'{str_task}/feature_columns.pkl')